# Home visitations — explanatory decision tree (INTEX)

**Purpose:** Understand what drives **Favorable** vs **non-favorable** home visit outcomes to draft a **Standard Operating Procedure (SOP)** checklist for social workers.

**Integration:** All training and JSON export code lives **in this notebook** (no separate `home_visit_explanatory` module). Run the **final code cell** to write `../backend/Models/home_visit_sop_insights.json` for the API and **Caseload Inventory**. For headless refresh, [`scripts/export_home_visit_sop.py`](scripts/export_home_visit_sop.py) mirrors the same logic. Only shared helper: [`helpers/pipeline_utils.py`](helpers/pipeline_utils.py) (`resolve_data_root`). **Not ONNX** — the site reads the JSON artifact.

**Notebook map**

| Section | Focus |
|--------|--------|
| 1 | Problem framing (explanatory vs predictive) |
| 2 | Data load, label, features, leakage |
| 3 | sklearn `Pipeline` + shallow `DecisionTreeClassifier` |
| 4 | Train/test metrics |
| 5 | `plot_tree`, importances, plain-English rules |
| 6 | Deployment: SOP narrative + export |


## 1) Problem framing

**Business problem:** Social workers need clarity on **when** a home visit is likely to be **safe and productive** with a resident's family, before and during scheduling.

**Model type — explanatory (not production prediction):** We prioritize **human-readable rules** and **validity of drivers** for an SOP checklist over maximizing holdout accuracy. A **shallow decision tree** is appropriate because splits read like a **flowchart** for staff training and supervision.

**Deliverable:** A **Next Best Action / SOP checklist** derived from the tree (see §6), deployed to the app as **JSON** for the caseload admin UI.


## 2) Data acquisition, preparation, and exploration

**Source:** `home_visitations.csv` via `pipeline_utils.resolve_data_root` (same export as the Lighthouse DB seed).

**Target (binary):** `y = 1` if `visit_outcome` is **Favorable** (case-insensitive); `y = 0` for **Unfavorable**, **Needs Improvement**, **Inconclusive**, or any other label.

**Feature engineering:** From `family_members_present`, derive **`has_parent`** and **`has_sibling`** if the text contains `"Parent"` / `"Sibling"` (case-insensitive substring).

**Predictors used:** `location_visited` (categorical), `safety_concerns_noted` (binary), `has_parent`, `has_sibling`.

**Leakage prevention:** We **do not** use outcome-adjacent fields that describe **during/after** the visit, e.g. `family_cooperation_level`, `observations`, `follow_up_needed`, `follow_up_notes` — otherwise the model would encode tautologies ("favorable when cooperation was high") useless for **pre-visit** SOP design.


In [ ]:
# Self-contained pipeline (INTEX). Run from `ml-pipelines/` so paths resolve.
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

from helpers.pipeline_utils import resolve_data_root

SEED = 42


def _boolish_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(np.int32)
    sl = s.astype(str).str.strip().str.lower()
    return (
        sl.map({"true": 1, "1": 1, "yes": 1, "false": 0, "0": 0, "no": 0, "": 0})
        .fillna(0)
        .astype(np.int32)
    )


def build_visit_outcome_label(visit_outcome: pd.Series) -> pd.Series:
    v = visit_outcome.astype(str).str.strip().str.lower()
    return (v == "favorable").astype(np.int32)


def engineer_family_flags(family_members_present: pd.Series) -> Tuple[pd.Series, pd.Series]:
    t = family_members_present.fillna("").astype(str).str.lower()
    has_parent = t.str.contains("parent", regex=False).astype(np.int32)
    has_sibling = t.str.contains("sibling", regex=False).astype(np.int32)
    return has_parent, has_sibling


def load_home_visitations_frame(data_root: Path | None = None) -> pd.DataFrame:
    root = data_root or resolve_data_root(Path.cwd())
    path = root / "home_visitations.csv"
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def prepare_xy(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    has_parent, has_sibling = engineer_family_flags(df["family_members_present"])
    safety = _boolish_series(df["safety_concerns_noted"])
    loc = df["location_visited"].fillna("Unknown").astype(str)
    y = build_visit_outcome_label(df["visit_outcome"])
    X = pd.DataFrame(
        {
            "location_visited": loc,
            "safety_concerns_noted": safety,
            "has_parent": has_parent,
            "has_sibling": has_sibling,
        }
    )
    return X, y


def build_pipeline(max_depth: int = 3) -> Pipeline:
    categorical = ["location_visited"]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical,
            ),
        ],
        remainder="passthrough",
    )
    clf = DecisionTreeClassifier(max_depth=max_depth, criterion="gini", random_state=SEED)
    return Pipeline([("prep", preprocessor), ("clf", clf)])


def _sop_item_from_feature(feat: str, importance: float, rank: int) -> Dict[str, Any]:
    f_low = feat.lower()
    if "safety" in f_low or "concern" in f_low:
        title, detail = "Safety screening", (
            "Before scheduling, confirm whether prior or contextual safety concerns are documented. "
            "The tree splits on safety flags early when they appear in historical patterns."
        )
    elif "parent" in f_low:
        title, detail = "Parental / guardian presence", (
            "Confirm a parent or legal guardian is expected or present when the visit plan indicates "
            "a parent role — structured guardian engagement aligns with more favorable visit patterns in this data."
        )
    elif "sibling" in f_low:
        title, detail = "Sibling presence", (
            "Note who else is in the home visit context; sibling presence is used as a structured "
            "signal for planning conversation dynamics, not as a fitness judgment."
        )
    elif "location" in f_low or "cat__" in f_low or "visited" in f_low:
        tail = feat.split("__")[-1] if "__" in feat else feat
        title, detail = "Location strategy", (
            f"Location category `{tail}` appears in the tree — align site choice with assessment type "
            "(neutral vs family home vs barangay office) per agency policy."
        )
    else:
        title = f"Factor {rank}: {feat[:60]}"
        detail = f"Model importance {importance:.3f}. This transformed input contributes to the explanatory splits."
    return {"step": rank, "title": title, "detail": detail, "feature": feat, "importance": float(importance)}


def build_sop_checklist(feature_names: np.ndarray, importances: np.ndarray) -> List[Dict[str, Any]]:
    pairs = sorted(zip(feature_names.tolist(), importances.tolist()), key=lambda x: -x[1])
    items: List[Dict[str, Any]] = []
    rank = 1
    for feat, imp in pairs:
        if imp <= 0:
            continue
        items.append(_sop_item_from_feature(feat, imp, rank))
        rank += 1
        if rank > 8:
            break
    anchors = [
        {
            "step": 0,
            "title": "Safety first",
            "detail": (
                "If safety concerns are documented for this address or household, require two staff "
                "or relocate to a neutral site per agency SOP before proceeding."
            ),
            "feature": "",
            "importance": 0.0,
        },
        {
            "step": 0,
            "title": "Confirm guardian attendance",
            "detail": (
                "When no parent/guardian is indicated in the visit plan, delay or reschedule until "
                "guardian attendance is confirmed unless emergency protocols apply."
            ),
            "feature": "",
            "importance": 0.0,
        },
    ]
    return anchors + items


def train_and_evaluate(
    X: pd.DataFrame, y: pd.Series, test_size: float = 0.2
) -> Tuple[Pipeline, Dict[str, Any], np.ndarray, np.ndarray]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=SEED
    )
    pipe = build_pipeline(max_depth=3)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = float(accuracy_score(y_test, y_pred))
    f1 = float(f1_score(y_test, y_pred, pos_label=1, zero_division=0))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    metrics = {
        "accuracy": acc,
        "f1PositiveClass": f1,
        "confusionMatrix": cm.tolist(),
        "confusionLabels": {
            "rows": ["actual_unfavorable", "actual_favorable"],
            "cols": ["pred_unfavorable", "pred_favorable"],
        },
        "testSupport": {"n": int(len(y_test)), "favorableRate": float(y_test.mean())},
    }
    return pipe, metrics, y_test.values, y_pred


def export_insights_payload(pipe: Pipeline, metrics: Dict[str, Any]) -> Dict[str, Any]:
    prep = pipe.named_steps["prep"]
    clf = pipe.named_steps["clf"]
    feat_names = prep.get_feature_names_out()
    imps = clf.feature_importances_
    tree_text = export_text(clf, feature_names=list(feat_names))
    importance_rows = [
        {"feature": str(a), "importance": float(b)}
        for a, b in sorted(zip(feat_names, imps), key=lambda x: -x[1])
        if b > 0
    ]
    business_rules: List[str] = []
    for line in tree_text.splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("---"):
            business_rules.append(stripped)
    business_rules = business_rules[:25]
    sop = build_sop_checklist(feat_names, imps)
    return {
        "version": "1",
        "generatedAtUtc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "problemSummary": (
            "Explanatory decision tree (max_depth=3, gini) on home visit outcomes to surface "
            "drivers of favorable vs non-favorable visits for social-worker SOP planning — "
            "interpretability over raw predictive accuracy."
        ),
        "modelParams": {"maxDepth": 3, "criterion": "gini", "randomState": SEED},
        "metrics": metrics,
        "featureImportances": importance_rows,
        "businessRules": business_rules,
        "treeText": tree_text,
        "sopChecklist": sop,
        "notes": (
            "Explanatory model for rubric discussion only. Refresh by re-running this notebook's export cell "
            "or scripts/export_home_visit_sop.py."
        ),
    }


def export_json_to_backend(
    backend_models_dir: Path | None = None,
    data_root: Path | None = None,
) -> Path:
    """Write backend/Models/home_visit_sop_insights.json. Expect cwd = ml-pipelines (or pass paths)."""
    nb_root = Path.cwd()
    if backend_models_dir is None:
        backend_models_dir = (nb_root.parent / "backend" / "Models").resolve()
    df = load_home_visitations_frame(data_root)
    X, y = prepare_xy(df)
    pipe, metrics, _, _ = train_and_evaluate(X, y)
    payload = export_insights_payload(pipe, metrics)
    backend_models_dir.mkdir(parents=True, exist_ok=True)
    out = backend_models_dir / "home_visit_sop_insights.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    return out


# --- Exploration (§2) ---
BASE_DIR = Path.cwd()
DATA_ROOT = resolve_data_root(BASE_DIR)

df = load_home_visitations_frame(DATA_ROOT)
print("rows:", len(df))
print("visit_outcome value counts:\n", df["visit_outcome"].astype(str).str.strip().value_counts().head(10))

X, y = prepare_xy(df)
print("\nFeature frame:", X.shape, "  favorable rate:", y.mean().round(3))
display(X.head())
display(
    pd.Series(
        {
            "has_parent": X["has_parent"].mean(),
            "has_sibling": X["has_sibling"].mean(),
            "safety_flag_rate": X["safety_concerns_noted"].mean(),
        }
    )
)


## 3) Modeling and feature selection

**Pipeline:** `ColumnTransformer` with **OneHotEncoder** (`handle_unknown='ignore'`) on `location_visited`; **passthrough** for `safety_concerns_noted`, `has_parent`, `has_sibling`.

**Estimator:** `DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)` — shallow on purpose to avoid an unreadable tree and overfitting memorization.


In [ ]:
pipe_preview = build_pipeline(max_depth=3)
pipe_preview


## 4) Evaluation and interpretation (generalization)

**Split:** 80/20 **stratified** holdout (`random_state=42`).

**Metrics:** Accuracy, F1 for the **positive class** (Favorable), and a **confusion matrix** on the test set — even for an explanatory model, we show that the learned rules are not pure training noise.


In [ ]:
pipe, metrics, y_test, y_pred = train_and_evaluate(X, y, test_size=0.2)
print("Accuracy:", round(metrics["accuracy"], 4))
print("F1 (favorable):", round(metrics["f1PositiveClass"], 4))
print("Confusion matrix [ [TN, FP], [FN, TP] ] labels=0,1 non-favorable,favorable:")
for row in metrics["confusionMatrix"]:
    print(row)
print("Test n:", metrics["testSupport"]["n"], " test favorable rate:", round(metrics["testSupport"]["favorableRate"], 3))


## 5) Causal and relationship analysis (visual + narrative)

We visualize the **fitted** tree with `plot_tree(..., filled=True)` using **transformed** feature names from the preprocessor. **Feature importances** summarize global split contribution. **Plain-English:** read **top-down** — the **root split** is the dominant factor in this shallow model (e.g. safety concerns before location).

*Caveat:* This is **associational** structure in observational data, not proof of causation; the section title follows course wording for interpretive analysis.


In [ ]:
from sklearn.tree import plot_tree

prep = pipe.named_steps["prep"]
clf = pipe.named_steps["clf"]
feat_names = prep.get_feature_names_out()

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(clf, feature_names=list(feat_names), class_names=["non_fav", "favorable"], filled=True, ax=ax, fontsize=9)
plt.title("Explanatory tree: home visit favorable vs not (max_depth=3)")
plt.tight_layout()
plt.show()

imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)
display(imp.head(12))


### Plain-English business rules (read from root downward)

Use the printed `export_text` below in supervision: each indentation is a **decision branch** leading to a **majority class** at the leaf. Relate branches to **Safety**, **Guardian presence**, and **Location strategy** when narrating the SOP.


In [ ]:
from sklearn.tree import export_text

print(export_text(clf, feature_names=list(feat_names)))


## 6) Deployment notes — Next Best Action SOP checklist

Translate the **root split** and the next one or two levels into **actionable pre-visit checks**:

1. **Safety first:** If safety concerns are documented for this household or site, **do not** send a single worker without escalation; use **two staff** or a **neutral location** per agency policy.
2. **Guardian confirmation:** When history shows visits without an indicated **parent/guardian** in `family_members_present`, align scheduling with **confirmed guardian attendance** unless a documented emergency exception applies.
3. **Location strategy:** Match **visit site** (e.g. barangay office vs family home vs foster home) to the **purpose** of the visit (routine follow-up vs reintegration assessment); use the tree's location branches to brief workers on where past favorable patterns clustered — **without** overriding clinical judgment.

**Production handoff:** Run the next cell to refresh `home_visit_sop_insights.json` consumed by `GET /api/Residents/home-visit-sop-insights` and the **Caseload Inventory** UI table.


In [ ]:
import json

# Re-fit stratified split and write ../backend/Models/home_visit_sop_insights.json (same logic as above cells)
out_path = export_json_to_backend()
print("Wrote:", out_path)

with open(out_path, encoding="utf-8") as f:
    written = json.load(f)
print("keys:", sorted(written.keys()))
